# OpenBCI 신경신호 수집 & 신호 처리 튜토리얼

## 학습 목표
- OpenBCI 보드와 Serial 통신으로 뇌파(EEG) 신경신호 수집
- 신호 처리: 필터링, RMS, 파워 스펙트럼 분석
- 실시간 데이터 시각화
- 신호 데이터 저장 및 분석

## 필요한 하드웨어
- **OpenBCI Ganglion 또는 Cyton 보드**
- **BLED112 Serial 동글** (COM 포트 연결)

## 학습 순서
1️⃣ 환경 설정
2️⃣ 포트 확인
3️⃣ 보드 연결
4️⃣ 실시간 데이터 수집
5️⃣ 신호 처리
6️⃣ 시각화 & 분석
7️⃣ 데이터 저장

---

**시작하려면 아래 '셀 1: 환경 설정'부터 위→아래 순서로 실행하세요!**

# 셀 1️⃣: 환경 설정

필요한 라이브러리를 import하고 기본 설정을 합니다.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 상대 경로 설정
current_dir = Path('.').resolve()
sys.path.insert(0, str(current_dir))

print("✅ 라이브러리 로드 완료")
print(f"📁 현재 디렉토리: {current_dir}")
print(f"🐍 Python 버전: {sys.version.split()[0]}")
print(f"📊 NumPy 버전: {np.__version__}")
print(f"📈 Pandas 버전: {pd.__version__}")

# 셀 2️⃣: 포트 확인

OpenBCI 보드가 연결된 COM 포트를 확인합니다.

**예상 결과:**
```
사용 가능한 포트:
  COM3 ← OpenBCI 동글
  COM1
```

In [ ]:
import serial.tools.list_ports

# 사용 가능한 포트 확인
ports = serial.tools.list_ports.comports()

print("\n🔍 사용 가능한 COM 포트:")
print("="*50)

if not ports:
    print("❌ 포트를 찾을 수 없습니다!")
    print("   → OpenBCI 동글이 연결되어 있는지 확인하세요")
else:
    for i, port in enumerate(ports, 1):
        print(f"  {i}. {port.device} - {port.description}")
    
    # 첫 번째 포트를 기본값으로 설정
    default_port = ports[0].device
    print(f"\n✅ 기본 포트: {default_port}")
    print("\n💡 팁: 다른 포트를 사용하려면 아래 셀에서 COM_PORT를 변경하세요")

# 셀 3️⃣: OpenBCI 보드 연결

OpenBCI 보드와 Serial 통신으로 연결합니다.

**⚠️ 주의:** 이 셀을 실행하기 전에:
- OpenBCI GUI가 종료되어 있는지 확인하세요
- 보드의 전원이 켜져 있는지 확인하세요

In [ ]:
# CustomSample 클래스 정의 (open_bci_v3.py 에서 가져올 수 없을 때용)
class OpenBCISample:
    def __init__(self, id, channels, aux):
        self.id = id
        self.channel_data = channels
        self.aux_data = aux

# COM 포트 설정
COM_PORT = "COM3"  # 🔧 필요시 변경: COM1, COM4 등
BAUDRATE = 115200   # 통신 속도

print(f"\n🔌 OpenBCI 보드 연결 중...")
print(f"   포트: {COM_PORT}")
print(f"   속도: {BAUDRATE} bps")
print(f"   대기: 3초...\n")

try:
    from open_bci_v3 import OpenBCIBoard
    
    # OpenBCI 보드 연결
    board = OpenBCIBoard(port=COM_PORT, baud=BAUDRATE, print_enable=True)
    
    print("\n✅ 보드 연결 성공!")
    print(f"   샘플링 레이트: {board.getSampleRate()} Hz")
    print(f"   EEG 채널: {board.getNbEEGChannels()}")
    print(f"   AUX 채널: {board.getNbAUXChannels()}")
    
except Exception as e:
    print(f"\n❌ 연결 실패: {e}")
    print("\n💡 해결 방법:")
    print("   1. OpenBCI GUI를 종료했는지 확인")
    print("   2. 보드가 켜져 있는지 확인")
    print("   3. COM_PORT 값을 올바른 포트로 변경")
    board = None

# 셀 4️⃣: 실시간 데이터 수집

보드에서 신경신호를 수집하고 실시간으로 표시합니다.

**수집 시간:** 10초 (조절 가능)

**표시 정보:**
- 패킷 ID (샘플 번호)
- 8개 채널의 전압값 (µV)
- 가속도계 데이터 (AUX)

In [ ]:
if board is None:
    print("❌ 보드가 연결되지 않았습니다. 셀 3을 다시 실행하세요.")
else:
    # 데이터 저장 버퍼
    collected_data = []
    sample_count = 0
    collection_duration = 10  # 🔧 수집 시간 (초)
    
    print(f"\n📊 {collection_duration}초간 데이터 수집 중...")
    print("="*70)
    
    def callback(sample):
        global sample_count, collected_data
        
        # 데이터 저장
        collected_data.append({
            'timestamp': sample_count / 250,  # 250Hz 샘플링
            'packet_id': sample.id,
            'ch1': sample.channel_data[0] if len(sample.channel_data) > 0 else 0,
            'ch2': sample.channel_data[1] if len(sample.channel_data) > 1 else 0,
            'ch3': sample.channel_data[2] if len(sample.channel_data) > 2 else 0,
            'ch4': sample.channel_data[3] if len(sample.channel_data) > 3 else 0,
            'ch5': sample.channel_data[4] if len(sample.channel_data) > 4 else 0,
            'ch6': sample.channel_data[5] if len(sample.channel_data) > 5 else 0,
            'ch7': sample.channel_data[6] if len(sample.channel_data) > 6 else 0,
            'ch8': sample.channel_data[7] if len(sample.channel_data) > 7 else 0,
        })
        
        sample_count += 1
        
        # 1초마다 진행 상황 출력 (250개 샘플 = 1초)
        if sample_count % 250 == 0:
            elapsed = sample_count / 250
            print(f"[{elapsed:.1f}s] {sample_count} 샘플 수집됨")
            print(f"      Ch1-4: {sample.channel_data[0]:8.2f}, {sample.channel_data[1]:8.2f}, {sample.channel_data[2]:8.2f}, {sample.channel_data[3]:8.2f} µV")
    
    try:
        # 스트리밍 시작
        board.start_streaming(callback, lapse=collection_duration)
        
        print("\n✅ 데이터 수집 완료!")
        print(f"   총 샘플 수: {sample_count}")
        print(f"   수집 시간: {sample_count/250:.2f}초")
        print("\n💾 데이터가 메모리에 저장되었습니다.")
        print("   → 다음 셀에서 신호 처리를 진행합니다.")
        
    except KeyboardInterrupt:
        print("\n⏸️  사용자가 중단했습니다")
    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")

# 셀 5️⃣: 신호 처리

수집한 데이터에 대해:
1. Butterworth 필터 적용 (5-50Hz 대역통과)
2. RMS (Root Mean Square) 계산
3. 파워 스펙트럼 분석

In [ ]:
from scipy import signal
from scipy.signal import welch

if len(collected_data) == 0:
    print("❌ 수집된 데이터가 없습니다. 셀 4를 먼저 실행하세요.")
else:
    # DataFrame으로 변환
    df = pd.DataFrame(collected_data)
    
    print("\n📈 신호 처리 중...")
    print("="*70)
    
    # 1. 필터 설계 (Butterworth 5-50Hz)
    fs = 250  # 샘플링 레이트
    low_freq = 5
    high_freq = 50
    order = 2
    
    nyquist = fs / 2
    normalized_low = low_freq / nyquist
    normalized_high = high_freq / nyquist
    
    b, a = signal.butter(order, [normalized_low, normalized_high], btype='band')
    
    print(f"\n🔧 필터 설정:")
    print(f"   필터 유형: Butterworth")
    print(f"   대역폭: {low_freq}-{high_freq} Hz")
    print(f"   차수: {order}")
    
    # 2. 각 채널에 필터 적용
    filtered_data = {}
    rms_values = {}
    
    print(f"\n📊 각 채널 분석:")
    print("-" * 70)
    print(f"{'채널':<8} {'RMS (µV)':<15} {'Max (µV)':<15} {'Min (µV)':<15}")
    print("-" * 70)
    
    for ch in range(1, 9):
        col_name = f'ch{ch}'
        raw = df[col_name].values
        
        # 필터 적용
        filtered = signal.filtfilt(b, a, raw)
        filtered_data[col_name] = filtered
        
        # RMS 계산
        rms = np.sqrt(np.mean(filtered**2))
        rms_values[col_name] = rms
        
        # 통계
        max_val = np.max(filtered)
        min_val = np.min(filtered)
        
        print(f"Ch{ch}     {rms:13.3f}   {max_val:13.3f}   {min_val:13.3f}")
    
    print("-" * 70)
    print("\n✅ 신호 처리 완료!")

# 셀 6️⃣: 시각화 & 분석

처리된 신호를 시각화합니다:
1. **채널별 시계열 파형**
2. **파워 스펙트럼 (주파수 영역)**

In [ ]:
if len(filtered_data) == 0:
    print("❌ 필터링된 데이터가 없습니다. 셀 5를 먼저 실행하세요.")
else:
    # 1️⃣ 채널별 파형
    fig, axes = plt.subplots(4, 2, figsize=(14, 10))
    fig.suptitle('OpenBCI 신경신호 - 시간 영역', fontsize=16, fontweight='bold')
    
    time_array = df['timestamp'].values
    
    for idx, ch in enumerate(range(1, 9)):
        ax = axes[idx // 2, idx % 2]
        col_name = f'ch{ch}'
        
        ax.plot(time_array, filtered_data[col_name], 'b-', linewidth=0.8)
        ax.set_xlabel('시간 (초)')
        ax.set_ylabel('전압 (µV)')
        ax.set_title(f'채널 {ch} (RMS: {rms_values[col_name]:.2f} µV)')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ 채널별 파형 표시 완료!")

# 셀 7️⃣: 파워 스펙트럼

각 채널의 주파수 영역 분석 (Welch 방법)

In [ ]:
if len(filtered_data) == 0:
    print("❌ 필터링된 데이터가 없습니다.")
else:
    # 파워 스펙트럼
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('OpenBCI 신경신호 - 파워 스펙트럼 (주파수 영역)', fontsize=16, fontweight='bold')
    
    channels_to_plot = [0, 1, 2, 3]  # Ch1-4만 표시
    
    for idx, ch_idx in enumerate(channels_to_plot):
        ax = axes[idx // 2, idx % 2]
        col_name = f'ch{ch_idx + 1}'
        
        # Welch 파워 스펙트럼
        freqs, pxx = welch(filtered_data[col_name], fs=250, nperseg=256)
        
        ax.semilogy(freqs, pxx, 'b-', linewidth=1.5)
        ax.set_xlabel('주파수 (Hz)')
        ax.set_ylabel('파워 (µV²/Hz)')
        ax.set_title(f'채널 {ch_idx + 1}')
        ax.grid(True, alpha=0.3, which='both')
        ax.set_xlim([0, 100])
    
    plt.tight_layout()
    plt.show()
    
    print("✅ 파워 스펙트럼 표시 완료!")

# 셀 8️⃣: 대역별 파워 분석

신경신호의 주요 대역:
- **Delta (0.5-4 Hz):** 깊은 수면
- **Theta (4-8 Hz):** 명상, 졸음
- **Alpha (8-12 Hz):** 휴식, 이완
- **Beta (12-30 Hz):** 각성, 집중
- **Gamma (30-100 Hz):** 고차 인지

In [ ]:
if len(filtered_data) == 0:
    print("❌ 필터링된 데이터가 없습니다.")
else:
    fs = 250
    
    # 대역 정의
    bands = {
        'Delta': (0.5, 4),
        'Theta': (4, 8),
        'Alpha': (8, 12),
        'Beta': (12, 30),
        'Gamma': (30, 100)
    }
    
    print("\n🎵 대역별 파워 분석 (채널 1-4)")
    print("="*80)
    
    band_data = {}
    
    for ch in range(1, 5):
        col_name = f'ch{ch}'
        print(f"\n📊 채널 {ch}:")
        print("-" * 80)
        print(f"{'대역':<15} {'주파수':<20} {'파워 (µV²)':<20} {'정규화 (%)':<20}")
        print("-" * 80)
        
        # 각 대역의 파워 계산
        freqs, pxx = welch(filtered_data[col_name], fs=fs, nperseg=256)
        
        band_powers = {}
        total_power = np.sum(pxx)
        
        for band_name, (low, high) in bands.items():
            mask = (freqs >= low) & (freqs <= high)
            power = np.sum(pxx[mask])
            normalized = (power / total_power) * 100 if total_power > 0 else 0
            band_powers[band_name] = power
            
            print(f"{band_name:<15} {low}-{high} Hz{'':<10} {power:18.3f}   {normalized:18.1f}%")
        
        band_data[f'ch{ch}'] = band_powers
    
    print("\n✅ 대역별 분석 완료!")
    print("\n💡 해석:")
    print("   - Alpha 대역이 높으면: 이완 상태")
    print("   - Beta 대역이 높으면: 각성/집중 상태")
    print("   - Theta 대역이 높으면: 명상/졸음 상태")

# 셀 9️⃣: 데이터 저장

수집한 데이터를 CSV와 NumPy 형식으로 저장합니다.

In [ ]:
if len(collected_data) == 0:
    print("❌ 저장할 데이터가 없습니다.")
else:
    # 출력 디렉토리 생성
    output_dir = Path('..') / 'outputs'
    output_dir.mkdir(exist_ok=True)
    
    # 타임스탐프
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. CSV 저장
    csv_file = output_dir / f"OpenBCI_Tutorial_{timestamp}.csv"
    df.to_csv(csv_file, index=False)
    
    print(f"✅ CSV 저장됨: {csv_file}")
    print(f"   행 수: {len(df)}")
    print(f"   열 수: {len(df.columns)}")
    
    # 2. NumPy 저장
    eeg_data = df[['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'ch8']].values
    npy_file = output_dir / f"OpenBCI_Tutorial_{timestamp}_eeg.npy"
    np.save(npy_file, eeg_data)
    
    print(f"\n✅ NumPy 저장됨: {npy_file}")
    print(f"   형태: {eeg_data.shape} (샘플 수 × 채널 수)")
    
    # 3. 요약 정보
    print(f"\n📋 수집 요약:")
    print(f"   총 샘플: {len(df)}")
    print(f"   수집 시간: {len(df)/250:.2f}초")
    print(f"   샘플링 레이트: 250 Hz")
    print(f"\n📁 저장 위치: {output_dir}")

# 셀 🔟: 보드 연결 해제

작업 완료 후 보드의 연결을 안전하게 해제합니다.

In [ ]:
if board is not None:
    try:
        board.disconnect()
        print("✅ 보드 연결 해제 완료")
        print("\n🎓 튜토리얼 완료!")
        print("\n다음 단계:")
        print("  1. 수집한 데이터를 분석하거나")
        print("  2. 신호 처리 파라미터를 변경하여 재실행")
    except Exception as e:
        print(f"❌ 연결 해제 실패: {e}")
else:
    print("ℹ️  보드가 연결되지 않았습니다.")

---

# 📚 학습 요약

이 튜토리얼에서 학습한 내용:

## 1️⃣ 하드웨어 통신
- Serial 통신으로 OpenBCI 보드 연결
- 데이터 패킷 구조 (33바이트): Start + ID + 8채널×3바이트 + AUX + End

## 2️⃣ 신호 처리
- **필터링:** Butterworth IIR 필터로 노이즈 제거
- **시간 영역:** RMS (실효값) 계산으로 신호 강도 측정
- **주파수 영역:** Welch 파워 스펙트럼으로 대역별 에너지 분석

## 3️⃣ 신경신호 대역
- 각 대역이 뇌 상태를 나타냄
- 신호 분석으로 사용자의 상태(이완/집중) 파악 가능

## 4️⃣ 데이터 관리
- CSV로 저장하여 Excel/Python에서 분석
- NumPy로 저장하여 빠른 로딩 및 행렬 연산

---

## 🚀 다음 도전 과제

### 초급
- [ ] 다양한 필터 파라미터 테스트
- [ ] 다양한 환경(조용함/시끄러움)에서 수집
- [ ] 개별 채널 분석

### 중급
- [ ] 실시간 필터링 구현
- [ ] 움직임 감지 알고리즘
- [ ] 다중 보드 동시 수집

### 고급
- [ ] 머신러닝으로 뇌 상태 분류
- [ ] BCI (Brain-Computer Interface) 제어
- [ ] 임상 데이터 분석

---

**작성:** 2026-07-23  
**대상:** UNIST 재활재생개론 학생  
**라이센스:** MIT